##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 SGLang 與 Gemma 2 交互

[Gemma](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開源語言模式。 Gemma 模型採用與創建 Gemini 模型相同的研究和技術構建而成，是文本到文本、僅限解碼器的大語言模型 (LLM)，提供英語版本，具有開放權重、預訓練變體和指令調整變體。
Gemma 模型非常適合各種文本生成任務，包括問答、總結和推論。它們相對較小的尺寸使得可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
[SGLang](https://github.com/sgl-project/sglang?tab=readme-ov-file) 是大型語言模型的服務framework。它提供了快速的後端 runtime 和靈活的前端語言，讓您可以控制和自訂模型互動。
在此 notebook 中，您將學習如何在 Google Colab 環境中使用 **SGLang** http 伺服器、後端 runtime 和前端語言以各種方式建立 prompt Gemma 2 模型。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Using_with_SGLang.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 選擇 Colab runtime
要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 Gemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### Setup Hugging Face

**在深入學習本教學之前，讓我們先設定一下「擁 Hugging Face 部」：**

1. **Hugging Face 帳戶：** 如果您還沒有帳戶，您可以點選[此處](https://huggingface.co/join) 建立免費的Hugging Face 帳戶。

2. **Hugging Face token：** 透過點選[此處](https://huggingface.co/settings/tokens) 產生Hugging Face 存取權限（最好是`write` 權限）token。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的 HF token

將您的 Hugging Face token 新增至 Colab Secrets manager 以安全地儲存它。
1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. 建立一個新的secret，名稱為`HF_TOKEN`。
3. 將 HF token 金鑰複製/貼上到 `HF_TOKEN` 的值輸入框中。
4. 切換左側的按鈕以允許notebook 存取secret。

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 安裝依賴項

首先，您必須安裝 SGLang 所需的軟體套件。

In [2]:
!pip install "sglang[all]"

# Install FlashInfer accelerated kernels
!pip install flashinfer -i https://flashinfer.ai/whl/cu121/torch2.4/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.8/436.8 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 53.9 MB/s eta 0:00:00
   ━━

## 概述

SGLang 提供快速的後端 runtime 和靈活的前端語言。為了展示使用 SGLang prompted Gemma 2 的不同方式，此 notebook 分為以下部分：1. 使用 SGLang 啟動 HTTP 伺服器。使用Python `requests` 至prompt Gemma 使用 SGLang 的本地生成APIs。
2. 在沒有 HTTP 伺服器的情況下將 SGLang 後端 inference 引擎設定為 prompt Gemma。
3. 使用 SGLang 的前端生成語言 prompt Gemma 並探索其一些功能。

## 1. 向執行 Gemma 2 的 SGLang 伺服器發送請求

在本部分中，您將啟動 HTTP 伺服器來使用 SGLang 執行 Gemma 2，並使用本機產生 API 端點將 prompt 傳送至模型。

### 啟動伺服器

可以透過在終端機中執行以下命令來啟動 SGLang 伺服器：
`python -m sglang.launch_server --model-path google/gemma-2-2b-it --port YOUR_PREFERRED_PORT`
在Colab 環境中，您必須將 SGLang 伺服器作為Python 子進程執行，並使用Python 的`subprocess` 套件管理其終止。 SGLang 提供了一些實用方法來為您抽象化這些細節。 `execute_shell_command` 函數允許您將伺服器作為Python 子進程啟動，而`wait_for_server` 函數則等待伺服器啟動並執行，然後才能向其發送請求。
您可以直接為`--model-path` 參數指定Gemma 2 的Hugging Face 儲存庫 ID。 SGLang 將從 Hugging Face 儲存庫下載必要的檔案來啟動伺服器。
您可以使用 `--port` 參數設定您選擇的任何連接埠來執行 SGLang。
在整個 notebook 中，`--mem-fraction-static` 設定為 0.6，以避免在 Colab 免費層上執行時出現 CUDA 記憶體不足錯誤。將 `--mem-fraction-static` 參數設為較低的值會減少 KV 快取記憶體池的記憶體使用量。請根據您的用例隨意嘗試不同的值。
**注意**：以下程式碼片段定義了一個執行 shell 指令並等待伺服器準備就緒的函數。稍後將在此 notebook 中重複使用此函數。

In [3]:
from sglang.utils import (
    execute_shell_command,
    wait_for_server,
    terminate_process,
)

def start_server():

  process = execute_shell_command(
  """
  python -m sglang.launch_server --model-path google/gemma-2-2b-it \
  --mem-fraction-static 0.6 \
  --port 9000
  """
  )

  wait_for_server("http://localhost:9000")

  return process

呼叫先前定義的`start_server`函數來啟動伺服器並取得伺服器進程的參考。
**注意**：伺服器啟動並執行需要 2 - 4 分鐘。

In [4]:
server_process = start_server()

伺服器現在已準備就緒，可以從此 notebook 內部透過 http://localhost:9000/ 存取伺服器。

### 使用 SGLang 的 Native Generation API 向 Gemma 2 發送請求

以下程式碼片段使用Python的`requests` library來呼叫 SGLang 在伺服器上的本機產生API，以將prompt傳送至Gemma-2。
您可以指定採樣參數的首選值，例如 `temperature`、`top_p` `max_new_tokens` 等。
有關 SGLang 支援的採樣參數的完整列表，請參閱 SGLang 的 [SGLang 執行時中的採樣參數](https://sgl-project.github.io/references/sampling_params.html) 指南。

若要從模型產生streaming回應，請指定附加鍵，`stream`在請求 json 中設定為`True`，並將`requests.post`的`stream`參數設為`True`。
SGLang 的[快速入門](https://sgl-project.github.io/start/send_request.html#Streaming) 文件中提供了 streaming 產生的範例。

In [5]:
import requests
import json

response = requests.post(
    "http://localhost:9000/generate",
    json={
        "text": "What is the age of earth?.",
        "sampling_params": {
            "temperature": 0.8,
        },
    },
)
print(json.dumps(response.json(), indent=2))

{
  "text": " \n\nI'm confused. \n\nIs it billions of years old?\n\nPlease explain. \n\n\nYou're right to be confused! It's a big number. Here's a breakdown:\n\n**Earth is about 4.54 \u00b1 0.05 billion years old.**\n\n* **Billions:** This means it's older than you and me, for sure! \n* **4.54 billion:**  This is the most precise estimate we have. \n* **\u00b1 0.05:** This means there's a range of 0.0",
  "meta_info": {
    "prompt_tokens": 8,
    "completion_tokens": 128,
    "completion_tokens_wo_jump_forward": 128,
    "cached_tokens": 1,
    "finish_reason": {
      "type": "length",
      "length": 128
    },
    "id": "1485c86977304adf92b2db1f77054a07"
  }
}


您可以使用`sglang.utils` 中的`terminate_process` 函數來停止伺服器。這相當於從終端機按 Ctrl+C 來停止伺服器。

In [6]:
terminate_process(server_process)

## 2.使用 SGLang 後端引擎離線批量inference

SGLang 提供了 inference 引擎，讓您可以直接與 Gemma 2 等本地模型交互，而無需 HTTP 伺服器。您可以使用它來建立自訂伺服器或離線批次inference。
在本節中，您將初始化inference 引擎以執行Gemma 2 並向其發送一批prompt。

### 使用 Gemma 2 初始化 SGLang inference 引擎

透過為 `model_path` 參數指定其 Hugging 儲存庫 ID，建立 `sglang.Engine` 類別的實例以執行 Gemma 2。

In [7]:
from sglang import Engine

llm = Engine(model_path="google/gemma-2-2b-it", mem_fraction_static=0.6)

### 使用 SGLang inference 引擎批量 prompting Gemma 2

您可以透過以下方式之一將 inference 的一批 prompt 傳送到 SGLang 引擎：
1. 非streaming同步調用
2. 串流同步調用
3. 非streaming非同步調用
4. 串流異步調用

您將在以下部分中探索如何使用 SGLang 引擎的同步生成函數對一批 prompt 執行 inference，從 Gemma 2 產生 streaming 和非 streaming 回應。
您可以參考 SGLang 的[離線引擎API](https://sgl-project.github.io/backend/offline_engine_api.html)指南來取得非同步回應產生的範例。

### Non-streaming synchronous prompting

Define a list of prompts to query Gemma 2 with.

In [8]:
prompts = [
    "Summarize what a galaxy is in three to four lines.",
    "List any 3 observatories in the world.",
]

使用inference 引擎的`generate` 函數從Gemma 2 產生一批非streaming 回應。將您先前定義的 prompt 列表和可選的採樣參數字典傳遞給此函數。此函數傳回模型對prompt批次的完整回應清單。

In [9]:
sampling_params = {"temperature": 0.1}
outputs = llm.generate(prompts, sampling_params)

for prompt, output in zip(prompts, outputs):
    print("=================================================================\n")
    print(f"Prompt: {prompt}\n\nGenerated text: {output['text']}\n")


Prompt: Summarize what a galaxy is in three to four lines.

Generated text: 

A galaxy is a vast collection of stars, gas, dust, and dark matter held together by gravity. It is a massive, gravitationally bound system that can range in size from a few hundred thousand to billions of stars. Galaxies come in various shapes and sizes, from spiral galaxies like our Milky Way, to elliptical galaxies, and irregular galaxies. 



Prompt: List any 3 observatories in the world.

Generated text: 

Here are 3 observatories in the world:

1. **Keck Observatory:** Located on Mauna Kea in Hawaii, the Keck Observatory is home to two of the world's largest optical/infrared telescopes.
2. **Very Large Telescope (VLT):** Located in the Atacama Desert of Chile, the VLT is a collection of four telescopes that work together to provide high-resolution images of distant objects.
3. **James Webb Space Telescope (JWST):** Launched in December 2021, the JWST is the largest and most powerful space telescope ever

### 串流同步prompting

若要從模型產生streaming 回應到先前定義的prompt 批次，請迭代`prompts` 並呼叫inference 引擎的`generate` 函數，並將附加參數`stream` 引擎的`generate` 函數，並將附加參數`stream` 設為@@P00033@。您可以透過迭代 `generate` 函數的回應來存取 streaming 回應中的每個區塊。

In [10]:
for prompt in prompts:
    print("\n===============================================================\n")
    print(f"\nPrompt: {prompt}\n")
    print("Generated text: \n", end="", flush=True)

    for chunk in llm.generate(prompt, sampling_params, stream=True):
        print(chunk["text"], end="", flush=True)




Prompt: Summarize what a galaxy is in three to four lines.

Generated text: 


A galaxy is a massive collection of stars, gas, dust, and dark matter held together by gravity. These vast structures range in size from a few tens of thousands to billions of light-years across. Galaxies are the building blocks of the universe, containing billions of stars and countless planets. They come in various shapes and sizes, from spiral galaxies like our own Milky Way to elliptical galaxies and irregular galaxies. 



Prompt: List any 3 observatories in the world.

Generated text: 


Here are 3 observatories in the world:

1. **Keck Observatory:** Located on Mauna Kea in Hawaii, the Keck Observatory is one of the world's most powerful optical/infrared telescopes.
2. **Very Large Telescope (VLT):** Located in the Atacama Desert of Chile, the VLT is a collection of four telescopes that work together to provide high-resolution images of distant objects.
3. **European Southern Observatory (ESO) Very

現在您可以關閉並清理 SGLang inference 引擎。

In [11]:
llm.shutdown()

W1105 16:28:30.518000 131977277298240 torch/_inductor/compile_worker/subproc_pool.py:126] SubprocPool unclean exit


## 3. 使用前端結構化產生語言（SGLang）進行推論

除了 HTTP 伺服器和離線後端引擎之外，SGLang 還提供了支援更多客製化和複雜prompting 工作流程的前端語言。
在以下部分中，您將探索如何使用 SGLang 的前端語言與Gemma 2 啟動多輪對話。您還將了解如何從 Gemma 2 取得 JSON 格式的回應。

### 啟動伺服器

首先，您必須使用 SGLang 啟動伺服器，並指定 Gemma 的 Hugging Face 儲存庫 ID 2。您可以使用介紹部分中定義的函數來啟動伺服器。

In [12]:
server_process = start_server()

使用 SGLang 提供的 `function` 裝飾器來定義函數，該函數接受您想要向模型詢問的幾個問題作為其參數。 `user` 函數用於將使用者的問題加入對話中。 `sglang.gen` 函數用於從模型產生響應，然後使用 `assistant` 函數將其附加到對話中。

函數 prompt 使用 `question_1` 建立模型，然後依序使用 `question_2` prompt 建立模型. 該模型預計會根據對話歷史回答`question_2`。

In [13]:
from sglang import function, user, assistant, gen, set_default_backend, RuntimeEndpoint

@function
def multi_turn_question(s, question_1, question_2):
    s += user(question_1)
    s += assistant(gen("answer_1", max_tokens=128))
    s += user(question_2)
    s += assistant(gen("answer_2", max_tokens=128))

### 連接到伺服器

透過指定 URL 使用 `sglang.set_default_backend` 連線到伺服器。

In [14]:
set_default_backend(RuntimeEndpoint("http://localhost:9000"))

### 發送多輪問題至Gemma 2

現在，您可以執行先前定義的 `multi_turn_question` 函數來從模型產生回應。

In [15]:
state = multi_turn_question.run(
    question_1="Who are the first humans to land on the moon?",
    question_2="Which country did they belong to?",
)

for m in state.messages():
  print(m["role"], ":", m["content"])

user : Who are the first humans to land on the moon?
assistant : The first humans to ever land on the moon were a team from the **Apollo 11 mission**:

* **Neil Armstrong**:  He became the first person to walk on the moon. His famous quote, "One small step for man, one giant leap for mankind," encapsulates the magnitude of thishistoric event.
* **Buzz Aldrin**: Aldrin was the second human to walk on the moon and stayed with Armstrong for several hours on the lunar surface. 

They landed on the moon on **July 20, 1969**, bringing back a wealth of lunar samples and photos that remain highly significant
user : Which country did they belong to?
assistant : The first people to land on the moon were part of **the United States**, often simply referred to as Americans.  They were a team from NASA, the National Aeronautics and Space Administration, the US government's space program. 



請注意對話的歷史記錄是如何保存的，並且模型回答了第二個問題作為對話的延續。

### 執行一批多輪問題

您也可以將字典列表傳遞給 `run_batch`（其鍵指定 `multi_turn_question` 函數的參數）將一組多輪問題批次到模型。

In [16]:
states = multi_turn_question.run_batch(
    [
        {
                "question_1": "Who are the first humans to land on moon?",
                "question_2": "Which country did they belong to ?",
            },
        {
                "question_1": "Who is the first human to reach space?",
                "question_2": "Which country did they belong to?",
        },
    ]
)

for state in states:
  print("\n===============================================================\n")
  for message in state.messages():
    print(message["role"], ":", message["content"])




user : Who are the first humans to land on moon?
assistant : The first humans to land on the Moon were **Neil Armstrong** and **Buzz Aldrin** of the Apollo 11 mission on **July 20, 1969.** 

user : Which country did they belong to ?
assistant : Neil Armstrong and Buzz Aldrin were from the **United States**. 



user : Who is the first human to reach space?
assistant : The first human to reach space was **Yuri Gagarin**. 

On April 12, 1961, he completed one orbit of Earth in the Soviet Vostok 1 spacecraft. This event marked a significant moment in the history of human exploration, paving the way for further advancements and spaceflight feats. 

user : Which country did they belong to?
assistant : Yuri Gagarin was from **Soviet Union** at the time. 



### JSON 解碼

您可以使用正規表示式 (regex) 指定模型產生的答案必須遵循的 JSON 架構。
使用 Gemma 定義函數，以 JSON 格式產生任何動物的特定資訊 2. 在 sglang.gen 函數的 regex 參數中指定正規表示式 JSON 架構。

In [17]:
character_regex = (
    r"""\{\n"""
    + r"""    "name": "[\w\d\s]{1,16}",\n"""
    + r"""    "type": "(Mammals|Birds|Fish|Reptiles|Amphibians|Invertebrates)",\n"""
    + r"""    "reproduction": "(Sexual|Asexual)",\n"""
    + r"""    "life expectancy": "[0-9]{1,2}",\n"""
    + r"""\}"""
)

@function
def animal_gen(s, name):
    s += name + " is an animal. Please fill in the following information about this animal.\n"
    s += gen("json_output", max_tokens=256, regex=character_regex)

使用任何動物的名稱作為輸入來執行函數，以取得 JSON 格式的特徵。

In [18]:
state = animal_gen.run(name="Fish")
print(state.text())

Fish is an animal. Please fill in the following information about this animal.
{
    "name": "Fish",
    "type": "Mammals",
    "reproduction": "Sexual",
    "life expectancy": "10",
}


終止伺服器進程。

In [ ]:
terminate_process(server_process)

這些只是如何使用 SGLang 的前端語言設計 Gemma 2 的 prompting 工作流程的幾個範例。要了解有關其功能的更多信息，您可以參考 SGLang 的[前端：結構化生成語言 (SGLang)](https://sgl-project.github.io/frontend/frontend.html) 指南。

恭喜！您已成功探索如何使用 SGLang 提供 Gemma 2，如何在 Colab 環境中使用 SGLang 後端 runtime 和前端語言執行。現在您可以在 SGLang 中嘗試更複雜的 prompting 工作流程，以與 Gemma 2 互動。